In [6]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q datasets huggingface_hub pillow tqdm

In [8]:
from huggingface_hub import login

login()

In [18]:
CLASS_MAPPING = {
    "airplane": "airliner",
    "automobile": "sports car",
    "bird": "goldfinch",
    "cat": "tabby",
    "deer": "red deer",
    "dog": "golden retriever",
    "frog": "tree frog",
    "horse": "sorrel",
    "ship": "container ship",
    "truck": "trailer truck"
}

SPLIT_CONFIG = {
    "train": 500,
    "val": 100,
    "test": 100
}

SAVE_DIR = "/content/drive/MyDrive/ImageNet10"

In [19]:
from datasets import load_dataset

# Load tập train của ImageNet-1K ở chế độ streaming
dataset = load_dataset(
    "ILSVRC/imagenet-1k",
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")
print(dataset.features)

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Dataset loaded successfully!
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['tench, Tinca tinca', 'goldfish, Carassius auratus', 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias', 'tiger shark, Galeocerdo cuvieri', 'hammerhead, hammerhead shark', 'electric ray, crampfish, numbfish, torpedo', 'stingray', 'cock', 'hen', 'ostrich, Struthio camelus', 'brambling, Fringilla montifringilla', 'goldfinch, Carduelis carduelis', 'house finch, linnet, Carpodacus mexicanus', 'junco, snowbird', 'indigo bunting, indigo finch, indigo bird, Passerina cyanea', 'robin, American robin, Turdus migratorius', 'bulbul', 'jay', 'magpie', 'chickadee', 'water ouzel, dipper', 'kite', 'bald eagle, American eagle, Haliaeetus leucocephalus', 'vulture', 'great grey owl, great gray owl, Strix nebulosa', 'European fire salamander, Salamandra salamandra', 'common newt, Triturus vulgaris', 'eft', 'spotted salamander, Ambystoma maculatum', 'axolotl, mud puppy, Ambys

In [20]:
CLASS_MAPPING = {
    "airplane": "airliner",
    "automobile": "sports car",
    "bird": "brambling",
    "cat": "tabby",
    "deer": "gazelle",
    "dog": "golden retriever",
    "frog": "tree frog",
    "horse": "sorrel",
    "ship": "container ship",
    "truck": "trailer truck"
}

label_names = dataset.features["label"].names

class_to_id = {}

for cifar_class, imagenet_class in CLASS_MAPPING.items():

    matches = [
        idx for idx, name in enumerate(label_names)
        if name.lower().split(", ")[0] == imagenet_class.lower()
    ]

    if matches:
        class_to_id[cifar_class] = matches[0]
        print(
            f"{cifar_class:12s} -> "
            f"{label_names[matches[0]]} "
            f"(ID: {matches[0]})"
        )
    else:
        print(f"Not found: {cifar_class}")

assert len(class_to_id) == 10
assert len(set(class_to_id.values())) == 10

print("\nMapping completed:", len(class_to_id), "classes")

airplane     -> airliner (ID: 404)
automobile   -> sports car, sport car (ID: 817)
bird         -> brambling, Fringilla montifringilla (ID: 10)
cat          -> tabby, tabby cat (ID: 281)
deer         -> gazelle (ID: 353)
dog          -> golden retriever (ID: 207)
frog         -> tree frog, tree-frog (ID: 31)
horse        -> sorrel (ID: 339)
ship         -> container ship, containership, container vessel (ID: 510)
truck        -> trailer truck, tractor trailer, trucking rig, rig, articulated lorry, semi (ID: 867)

Mapping completed: 10 classes


In [21]:
import os
from PIL import Image
from tqdm.auto import tqdm

from collections import defaultdict

# Tạo thư mục
for split in SPLIT_CONFIG:
    for class_name in class_to_id:
        os.makedirs(
            f"{SAVE_DIR}/{split}/{class_name}",
            exist_ok=True
        )

# Tổng ảnh cần lấy mỗi class
TOTAL_PER_CLASS = sum(SPLIT_CONFIG.values())

counts = defaultdict(int)

id_to_class = {
    label_id: class_name
    for class_name, label_id in class_to_id.items()
}

total_needed = len(class_to_id) * TOTAL_PER_CLASS

pbar = tqdm(total=total_needed)

for sample in dataset:

    label_id = sample[label_key]

    if label_id not in id_to_class:
        continue

    class_name = id_to_class[label_id]

    if counts[class_name] >= TOTAL_PER_CLASS:
        continue

    image = sample["image"].convert("RGB")

    index = counts[class_name]

    # Xác định split
    if index < SPLIT_CONFIG["train"]:
        split = "train"

    elif index < SPLIT_CONFIG["train"] + SPLIT_CONFIG["val"]:
        split = "val"

    else:
        split = "test"

    image_path = (
        f"{SAVE_DIR}/{split}/{class_name}/"
        f"{index:05d}.jpg"
    )

    image.save(image_path, quality=95)

    counts[class_name] += 1
    pbar.update(1)

    if all(
        counts[c] >= TOTAL_PER_CLASS
        for c in class_to_id
    ):
        break

pbar.close()

print("Download completed!")
print(dict(counts))

  0%|          | 0/7000 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


Download completed!
{'deer': 700, 'airplane': 700, 'frog': 700, 'cat': 700, 'dog': 700, 'ship': 700, 'truck': 700, 'automobile': 700, 'horse': 700, 'bird': 700}


In [22]:
for split, expected in SPLIT_CONFIG.items():

    print(f"\n{split.upper()}")

    for class_name in class_to_id:

        folder = f"{SAVE_DIR}/{split}/{class_name}"

        n_images = len([
            f for f in os.listdir(folder)
            if f.endswith(".jpg")
        ])

        print(f"{class_name}: {n_images}")


TRAIN
airplane: 500
automobile: 500
bird: 500
cat: 500
deer: 500
dog: 500
frog: 500
horse: 500
ship: 500
truck: 500

VAL
airplane: 100
automobile: 100
bird: 100
cat: 100
deer: 100
dog: 100
frog: 100
horse: 100
ship: 100
truck: 100

TEST
airplane: 100
automobile: 100
bird: 100
cat: 100
deer: 100
dog: 100
frog: 100
horse: 100
ship: 100
truck: 100
